# 04 — Grounded RAG and Prompt Engineering

This notebook connects the completed retriever to a local Hugging Face instruction model. It demonstrates the complete RAG path: **question → retrieve → budget context → prompt → generate → attach source provenance**.

Two Hugging Face models have different responsibilities:

- `all-MiniLM-L6-v2`: embeds questions and chunks for retrieval.
- `Qwen2.5-0.5B-Instruct`: reads retrieved evidence and generates an answer.

Changing the generator does not require rebuilding the embedding index.

In [1]:
from pathlib import Path
import os, sys, logging
import numpy as np
import pandas as pd
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embed_retrieve import read_jsonl, load_embedding_model, retrieve_top_k
from src.rag_generate import load_generator, build_prompt, run_rag

embedding_model = load_embedding_model(PROJECT_ROOT / 'models/all-MiniLM-L6-v2')
chunks = read_jsonl(PROJECT_ROOT / 'data/processed/chunks_128.jsonl')
embeddings = np.load(PROJECT_ROOT / 'data/embeddings/embeddings_128.npy')
generator_tokenizer, generator_model = load_generator(
    PROJECT_ROOT / 'models/qwen2.5-0.5b-instruct'
)
print('Retriever:', 'all-MiniLM-L6-v2', embeddings.shape)
print('Generator:', 'Qwen2.5-0.5B-Instruct')
print('Device:', generator_model.device)

Retriever: all-MiniLM-L6-v2 (972, 384)
Generator: Qwen2.5-0.5B-Instruct
Device: cpu


## Checkpoint 1 — Retrieval remains separate from generation

`retrieve_top_k()` knows nothing about the LLM. It returns ranked dictionaries containing similarity, text, document and PDF page. This separation lets retrieval be tested independently and lets the generator be replaced without re-embedding documents.

In [2]:
question = 'Which measurable units can be used to schedule preventive maintenance?'
retrieved = retrieve_top_k(question, embedding_model, embeddings, chunks, top_k=3)
pd.DataFrame([{
    'rank': r['rank'], 'cosine_similarity': r['score'],
    'document': r['document'], 'pdf_page': r['pdf_page'],
    'text_preview': r['text'][:150]
} for r in retrieved]).style.format({'cosine_similarity': '{:.3f}'})

,rank,cosine_similarity,document,pdf_page,text_preview
0,1,0.677,NIST.AMS.100-34,15,2.2. Preventive Maintenance Preventive Maintenance (PM) is a strategy focused on performing specific maintenance routines based upon a specified inter
1,2,0.603,NIST.AMS.100-18,10,NIST efforts in maintenance research seeks to create standards that reduce the costs and losses associated with maintenance in manufacturing environme
2,3,0.585,NIST.AMS.100-18,15,"Table 2-2: Characteristics of Maintenance by Type Maintenance Type Reactive Preventive Predictive Frequency On Demand Scheduled, Timed, or Cycle Based"


## Checkpoint 2 — Context optimization is a token-budget decision

`build_prompt()` attempts to add complete chunks in rank order. Before accepting another chunk, it tokenizes the **entire chat-formatted prompt**. It stops rather than silently truncating evidence. The output records how many requested sources actually fit and the final input-token count.

This is why `top_k=3` and `sources_in_prompt=3` are different concepts: retrieval may request three chunks, but the generator budget decides how many can safely be passed onward.

In [3]:
prompt, selected, input_tokens = build_prompt(
    question, retrieved, generator_tokenizer,
    style='grounded', max_input_tokens=1024
)
print('Requested Top-k:', len(retrieved))
print('Complete sources placed in prompt:', len(selected))
print('Final generator input tokens:', input_tokens)
print('\nPROMPT PREVIEW\n', '-' * 100)
print(prompt[:1800])

Requested Top-k: 3
Complete sources placed in prompt: 3
Final generator input tokens: 526

PROMPT PREVIEW
 ----------------------------------------------------------------------------------------------------
Answer using only the evidence below in no more than three concise sentences. State the answer directly, do not discuss these instructions, and do not use outside knowledge. Source citations are attached separately by the system.

EVIDENCE:
[S1 | NIST.AMS.100-34, PDF p.15]
2.2.
Preventive Maintenance
Preventive Maintenance (PM) is a strategy focused on performing specific maintenance routines based upon a specified interval unit(s) - often time- or usage-based.
A common example that has been prevalent is the guidance for an automobile owner to change the vehicle’s oil every 3 months or 3000 miles (4828 kilometers) driven (whichever comes first).
The units can be easily monitored as time in months or distance in miles (or kilometers).
Similarly, there are numerous intervals that man

## Checkpoint 3 — Generation is deterministic; retrieved sources come from metadata

Generation uses greedy decoding (`do_sample=False`) so repeated experiments are comparable. Testing showed that small local models could generate a useful answer but did not reliably copy citation labels. Therefore the application retains all document/page metadata for the complete chunks actually placed in the prompt, deduplicates repeated pages, and appends them under `Retrieved sources`.

This list records the evidence available to the model; it does not claim that every listed passage was used for every sentence. Stage 7 must still check whether the answer is supported by the retrieved evidence.

In [4]:
supported_questions = [
    question,
    'What safety and maintainability benefits are associated with prognostics and health management?'
]
supported_runs = []
for item in supported_questions:
    result = run_rag(
        item, embedding_model, embeddings, chunks,
        generator_tokenizer, generator_model,
        top_k=3, prompt_style='grounded'
    )
    supported_runs.append(result)
    print('\n', '=' * 100)
    print('QUESTION:', item)
    print('ANSWER:', result['answer'])
    print('Top retrieval similarity:', round(result['top_similarity'], 3))
    print('Unique retrieved source pages:', len(result['retrieved_sources']))
    print('Input tokens / sources:', result['input_tokens'], '/', result['sources_in_prompt'])
    print('Generation latency:', round(result['latency_seconds'], 2), 'seconds')


QUESTION: Which measurable units can be used to schedule preventive maintenance?
ANSWER: Based on the provided information, the measurable units that can be used to schedule preventive maintenance include:

1. Time (months)
2. Distance (miles)
3. Cycles (cycles)

These units allow for monitoring and scheduling maintenance activities according to predefined intervals.

Retrieved sources:
1. NIST.AMS.100-34 — PDF page 15
2. NIST.AMS.100-18 — PDF page 10
3. NIST.AMS.100-18 — PDF page 15
Top retrieval similarity: 0.677
Unique retrieved source pages: 3
Input tokens / sources: 526 / 3
Generation latency: 8.92 seconds



QUESTION: What safety and maintainability benefits are associated with prognostics and health management?
ANSWER: Safety and maintainability benefits associated with prognostics and health management include advanced warning of failure, elimination of redundant inspections, minimized unscheduled maintenance, and decreased test equipment requirement.

Retrieved sources:
1. NIST.AMS.100-18 — PDF page 17
2. NIST.AMS.100-2 — PDF page 7
3. NIST.AMS.100-2 — PDF page 10
Top retrieval similarity: 0.806
Unique retrieved source pages: 3
Input tokens / sources: 321 / 3
Generation latency: 5.38 seconds


## Checkpoint 4 — Prompt instructions alone are not a sufficient refusal mechanism

A retriever always returns its nearest chunks—even when none really answers the question. The grounded pipeline rejects generation when the top cosine similarity is below the current `0.58` MVP threshold. This value was calibrated during Stage 7; it is not a probability and requires validation on new unsupported questions.

The comparison below shows why this system guardrail matters: the basic path sends weak context to the model, while the grounded path stops before generation.

In [5]:
unsupported_question = 'What lubricant viscosity should be used for a Sandisk assembly machine?'
comparison = []
for style in ('basic', 'grounded'):
    result = run_rag(
        unsupported_question, embedding_model, embeddings, chunks,
        generator_tokenizer, generator_model, top_k=3, prompt_style=style
    )
    comparison.append({
        'prompt_style': style,
        'top_similarity': result['top_similarity'],
        'evidence_gate_triggered': result['evidence_gate_triggered'],
        'generation_latency': result['latency_seconds'],
        'answer': result['answer']
    })
pd.DataFrame(comparison).style.format({
    'top_similarity': '{:.3f}', 'generation_latency': '{:.2f}'
})

,prompt_style,top_similarity,evidence_gate_triggered,generation_latency,answer
0,basic,0.336,False,9.10,"Based on the information provided in the given text, there is no specific mention of lubricant viscosity for a Sandisk assembly machine. The text does not provide any details about the viscosity requirements for such a machine. Therefore, I cannot answer the question with certainty based solely on the information given. Retrieved sources: 1. NIST.AMS.100-2 — PDF page 22 2. NIST.AMS.100-34 — PDF page 3 3. NIST.AMS.100-34 — PDF page 16"
1,grounded,0.336,True,0.00,INSUFFICIENT EVIDENCE


## Engineering conclusion

The first end-to-end RAG pipeline now works locally. Its key guardrails are:

1. retrieve from the evaluated 128-token index;
2. add complete sources within an explicit generator-token budget;
3. reject low-similarity evidence before generation;
4. generate deterministically from supplied context;
5. append all deduplicated document/page sources that were actually supplied to the generator.

Stage 7 evaluates Top-3 versus Top-5, basic versus grounded behavior, supported-answer points, retrieved-source coverage, unsupported-question rejection and latency.